# CholecSeg8k Evaluation

Evaluate the surgical annotation pipeline on the CholecSeg8k dataset with real metrics: mAP, precision, recall, F1.

**Prerequisites (from previous notebook):**
1. GPU enabled (T4)
2. Dependencies installed (transformers, sam2, opencv, etc.)
3. Project uploaded to `/content/surgical-video-annotator-v2`
4. Working directory changed to project folder

**New files needed:**
- `src/dataset_loader.py`
- `src/image_pipeline.py`

## Step 1: Install datasets library

In [ ]:
!pip install -q datasets

## Step 2: Verify new files are uploaded

In [ ]:
import os
required = ['src/dataset_loader.py', 'src/image_pipeline.py']
for f in required:
    if os.path.exists(f):
        print(f'OK: {f}')
    else:
        print(f'MISSING: {f} - upload it to /content/surgical-video-annotator-v2/src/')

## Step 3: Download CholecSeg8k subset

Start with 200 images. Downloads ~200MB, takes 3-5 minutes.

In [ ]:
import sys
sys.path.insert(0, '.')

from src.dataset_loader import CholecSeg8kLoader, CHOLECSEG8K_CLASSES

loader = CholecSeg8kLoader(cache_dir='cholecseg8k_data')
samples = loader.download_subset(n=200)

print(f'\nDownloaded {len(samples)} samples')
print(f'Images: {loader.get_image_dir()}')

print(f'\nDataset classes:')
for cid, name in CHOLECSEG8K_CLASSES.items():
    print(f'  {cid}: {name}')

## Step 4: Preview a sample

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

sample = samples[10]
image = Image.open(sample['image_path'])
mask = Image.open(sample['mask_path'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(image)
axes[0].set_title(f'Image: {os.path.basename(sample["image_path"])}')
axes[0].axis('off')

axes[1].imshow(mask)
axes[1].set_title('Ground Truth Mask')
axes[1].axis('off')
plt.tight_layout()
plt.show()

bboxes = loader.mask_to_bboxes(sample['mask_path'])
print(f'\nGround truth objects:')
for bbox in bboxes:
    print(f'  {bbox["class_name"]} at {bbox["bbox"]} ({bbox["pixel_count"]} pixels)')

## Step 5: Export ground truth as COCO JSON

In [ ]:
gt_path = loader.export_ground_truth_coco(
    output_path='cholecseg8k_gt.json',
    include_anatomy=True,
)

import json
with open(gt_path, 'r') as f:
    gt = json.load(f)

print(f'\nGround truth COCO summary:')
print(f'  Images: {len(gt["images"])}')
print(f'  Total annotations: {len(gt["annotations"])}')
print(f'  Categories: {len(gt["categories"])}')

from collections import Counter
class_counts = Counter(a['category_id'] for a in gt['annotations'])
cat_names = {c['id']: c['name'] for c in gt['categories']}

print(f'\nAnnotations per class:')
for cid, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f'  {cat_names[cid]}: {count}')

## Step 6: Get prompts for detection

In [ ]:
prompts_for_eval = loader.get_prompts_for_evaluation(include_anatomy=True)
print(f'Prompts:')
for p in prompts_for_eval:
    print(f'  - {p}')

## Step 7: Run image-mode pipeline with evaluation

Detects instruments and organs on all 200 images, then compares to ground truth.

Expected time: 5-10 minutes.

In [ ]:
from src.image_pipeline import ImageModePipeline

pipeline = ImageModePipeline('config.yaml')

results = pipeline.run(
    image_dir=loader.get_image_dir(),
    ground_truth_path='cholecseg8k_gt.json',
    prompts=prompts_for_eval,
    output_dir='output_evaluation',
    max_images=200,
    include_masks=False,
)

## Step 8: Review the metrics

In [ ]:
print('=' * 70)
print('PIPELINE STATS')
print('=' * 70)
for key, val in results['stats'].items():
    if key != 'evaluation':
        print(f'  {key}: {val}')

eval_result = results['evaluation']
if eval_result:
    print('\n' + '=' * 70)
    print('EVALUATION METRICS (against CholecSeg8k ground truth)')
    print('=' * 70)
    
    result_dict = eval_result.to_dict()
    
    print('\nOverall metrics:')
    for key, val in result_dict['summary'].items():
        print(f'  {key}: {val}')
    
    print(f'\nCounts:')
    for key, val in result_dict['counts'].items():
        print(f'  {key}: {val}')
    
    print(f'\nPer-class AP@0.5 (higher = better):')
    for cls, ap in sorted(result_dict['per_class']['ap_50'].items(), key=lambda x: -x[1]):
        print(f'  {cls}: {ap:.3f}')
    
    print(f'\nPer-class Precision:')
    for cls, p in sorted(result_dict['per_class']['precision'].items(), key=lambda x: -x[1]):
        print(f'  {cls}: {p:.3f}')
    
    print(f'\nPer-class Recall:')
    for cls, r in sorted(result_dict['per_class']['recall'].items(), key=lambda x: -x[1]):
        print(f'  {cls}: {r:.3f}')

## Step 9: Visualize predictions vs ground truth

In [ ]:
import cv2
import matplotlib.pyplot as plt
import json

with open('cholecseg8k_gt.json', 'r') as f:
    gt = json.load(f)
with open('output_evaluation/predictions_coco.json', 'r') as f:
    pred = json.load(f)

gt_by_image = {}
for a in gt['annotations']:
    gt_by_image.setdefault(a['image_id'], []).append(a)

pred_by_image = {}
for a in pred['annotations']:
    pred_by_image.setdefault(a['image_id'], []).append(a)

gt_cats = {c['id']: c['name'] for c in gt['categories']}
pred_cats = {c['id']: c['name'] for c in pred['categories']}

def draw_boxes(img, annotations, categories, color):
    for ann in annotations:
        x, y, w, h = [int(v) for v in ann['bbox']]
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
        label = categories.get(ann['category_id'], '?')
        cv2.putText(img, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

sample_ids = list(gt_by_image.keys())[:4]
fig, axes = plt.subplots(len(sample_ids), 2, figsize=(16, 5 * len(sample_ids)))

for i, img_id in enumerate(sample_ids):
    img_info = next((im for im in gt['images'] if im['id'] == img_id), None)
    if not img_info:
        continue
    img_path = os.path.join(loader.get_image_dir(), img_info['file_name'])
    img_gt = cv2.imread(img_path)
    img_pred = img_gt.copy()
    
    img_gt = draw_boxes(img_gt, gt_by_image.get(img_id, []), gt_cats, (0, 255, 0))
    img_pred = draw_boxes(img_pred, pred_by_image.get(img_id, []), pred_cats, (0, 0, 255))
    
    axes[i, 0].imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f'Ground Truth (image {img_id})')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(f'Pipeline Predictions')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

## Step 10: How to interpret the results

**What scores mean:**
- mAP@0.5 > 0.5: Great for zero-shot foundation models
- mAP@0.5 = 0.3-0.5: Decent, fine-tuning would help
- mAP@0.5 < 0.3: Need specialized detector

**What to look for:**
- Instruments (grasper, hook) should score higher than organs
- Organs will likely score low (foundation models weren't trained on surgical anatomy)
- Wide variation between classes reveals which prompts work

**Next steps based on results:**
- If mAP > 0.5: Pipeline is working well, add masks and demo it
- If instruments good, organs bad: Focus demo on instruments
- If everything low: Move to specialized surgical detector (Option A)

## Step 11: Download results

In [ ]:
!zip -r cholecseg8k_evaluation.zip output_evaluation/ cholecseg8k_gt.json

from google.colab import files
files.download('cholecseg8k_evaluation.zip')